In [ ]:
!pip install rasterio numpy tensorflow keras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Módulo 1 — Validação





In [ ]:
# ============================================================
# MÓDULO 1 — Validação dos dados
# ============================================================
# Objetivo: verificar se todos os pares (imagem + máscara)
# estão corretos antes de treinar a U-Net.
#
# Pense neste módulo como um check-up médico dos seus dados.
# Se algo estiver errado aqui, melhor descobrir agora do que
# depois de horas de treino.
# ============================================================

import os
import numpy as np
import rasterio

# ----------------------------------------------------------
# CONFIGURAÇÃO — ajuste só os caminhos das pastas
# ----------------------------------------------------------
TAMANHO_ESPERADO = 128

PASTA_IMAGENS  = "/content/drive/MyDrive/ic/imagem"
PASTA_MASCARAS = "/content/drive/MyDrive/ic/mask"

# Percorre as pastas automaticamente
imgs  = {f for f in os.listdir(PASTA_IMAGENS)  if f.endswith(".tif")}
masks = {f for f in os.listdir(PASTA_MASCARAS) if f.endswith(".tif")}

# ── GAP 3 CORRIGIDO ─────────────────────────────────────────
# Antes: int(nome.replace('.tif', '')) → quebra com "1_old.tif"
# Agora: try/except pula arquivos com nome inesperado e avisa
def extrair_numero(nome_arquivo):
    try:
        return int(nome_arquivo.replace(".tif", ""))
    except ValueError:
        print(f"  ⚠ Nome inesperado ignorado: '{nome_arquivo}' — esperado só número (ex: '1.tif')")
        return None

# Filtra apenas arquivos cujo nome é um número válido
imgs  = {f for f in imgs  if extrair_numero(f) is not None}
masks = {f for f in masks if extrair_numero(f) is not None}
# ────────────────────────────────────────────────────────────

nomes_validos = sorted(imgs & masks, key=extrair_numero)
sem_mascara   = sorted(imgs - masks, key=extrair_numero)
sem_imagem    = sorted(masks - imgs, key=extrair_numero)

if sem_mascara:
    print(f"⚠ Imagens sem máscara: {sem_mascara}")
if sem_imagem:
    print(f"⚠ Máscaras sem imagem: {sem_imagem}")

PARES = [
    {
        "imagem" : os.path.join(PASTA_IMAGENS,  nome),
        "mascara": os.path.join(PASTA_MASCARAS, nome),
    }
    for nome in nomes_validos
]

print(f"  {len(PARES)} par(es) encontrado(s): {nomes_validos}")
# ----------------------------------------------------------


def checar_crs(ds_imagem, ds_mascara) -> bool:
    """
    Verifica se os dois arquivos estão no mesmo sistema de coordenadas.

    CRS (Coordinate Reference System) é como o GPS do arquivo —
    define em que projeção o mapa está. Se forem diferentes,
    um estará em graus e o outro em metros, por exemplo, e
    os pixels não vão se alinhar corretamente.
    """
    ok = ds_imagem.crs == ds_mascara.crs

    print("── CRS ─────────────────────────────────────")
    print(f"  Imagem : {ds_imagem.crs}")
    print(f"  Máscara: {ds_mascara.crs}")
    print(f"  {'✓ Iguais — ok' if ok else '✗ DIFERENTES — vai causar desalinhamento!'}")
    return ok


def checar_resolucao(ds_imagem, ds_mascara) -> bool:
    """
    Verifica se os pixels têm o mesmo tamanho nos dois arquivos.

    A resolução espacial do Sentinel-2 RGB é 10 m/pixel.
    A máscara precisa ter exatamente o mesmo tamanho de pixel
    para que cada pixel da imagem corresponda a exatamente um
    pixel da máscara — como um molde de cookie encaixando na massa.
    """
    ok = np.allclose(ds_imagem.res, ds_mascara.res, rtol=1e-3)

    print("\n── Resolução espacial ──────────────────────")
    print(f"  Imagem : {ds_imagem.res[0]:.2f} m × {ds_imagem.res[1]:.2f} m")
    print(f"  Máscara: {ds_mascara.res[0]:.2f} m × {ds_mascara.res[1]:.2f} m")
    print(f"  {'✓ Compatíveis — ok' if ok else '✗ DIFERENTES — precisará de re-amostragem.'}")
    return ok


# ── GAP 4 — NOVA FUNÇÃO ─────────────────────────────────────
# Verifica se imagem e máscara cobrem exatamente a mesma
# área geográfica, comparando os bounds (bordas em coordenadas).
#
# Por que isso importa?
# Imagine dois recortes de mapa que têm o mesmo CRS e o mesmo
# tamanho de pixel, mas um começa em x=300000 e o outro em
# x=300005. Os pixels estariam deslocados 5 metros — a rede
# aprenderia a associar plantação com o lugar errado.
#
# ⚠ Linha crítica: a tolerância é 0.01 m (1 cm).
#   Se quiser ser mais permissivo, aumente para 0.5 m, por exemplo.
#   Mas com dados bem exportados do QGIS, 0.01 m funciona.
def checar_extensao_geografica(ds_imagem, ds_mascara) -> bool:
    """
    Compara os bounds (extensão geográfica) de imagem e máscara.
    Tolerância: 0.01 m em cada borda.
    """
    b_img  = ds_imagem.bounds
    b_mask = ds_mascara.bounds

    tolerancia = 0.01
    ok = (
        abs(b_img.left   - b_mask.left)   <= tolerancia and
        abs(b_img.right  - b_mask.right)  <= tolerancia and
        abs(b_img.top    - b_mask.top)    <= tolerancia and
        abs(b_img.bottom - b_mask.bottom) <= tolerancia
    )

    print("\n── Extensão geográfica (bounds) ────────────")
    print(f"  Imagem : {b_img}")
    print(f"  Máscara: {b_mask}")
    print(f"  {'✓ Alinhados — ok' if ok else '✗ DESALINHAMENTO — checar no QGIS!'}")
    return ok
# ────────────────────────────────────────────────────────────


# ── GAP 1 CORRIGIDO ─────────────────────────────────────────
# Antes: só checava se imagem e máscara tinham o mesmo tamanho
#        entre si — não verificava se eram 128×128
# Agora: rejeita o par se H ou W for diferente de TAMANHO_ESPERADO
def checar_dimensoes(arr_imagem: np.ndarray, arr_mascara: np.ndarray) -> bool:
    """
    Verifica se imagem e máscara têm exatamente 128×128 pixels.

    Mesmo que CRS e resolução batam, a área coberta pode ser
    levemente diferente, gerando grades de tamanhos distintos.
    A U-Net espera entradas com tamanho fixo — se chegar 130×128,
    vai dar erro na camada de entrada.
    """
    _, H_img, W_img   = arr_imagem.shape
    _, H_mask, W_mask = arr_mascara.shape

    tamanho_ok = (
        H_img  == TAMANHO_ESPERADO and
        W_img  == TAMANHO_ESPERADO and
        H_mask == TAMANHO_ESPERADO and
        W_mask == TAMANHO_ESPERADO
    )
    iguais = (H_img == H_mask) and (W_img == W_mask)
    ok = tamanho_ok and iguais

    print("\n── Dimensões (H × W) ───────────────────────")
    print(f"  Imagem : {H_img} × {W_img}  (esperado: {TAMANHO_ESPERADO} × {TAMANHO_ESPERADO})")
    print(f"  Máscara: {H_mask} × {W_mask}")
    if not tamanho_ok:
        print(f"  ✗ Tamanho diferente de {TAMANHO_ESPERADO}×{TAMANHO_ESPERADO} — não será usado no treino.")
    elif not iguais:
        print("  ✗ Imagem e máscara com tamanhos diferentes entre si.")
    else:
        print("  ✓ Corretos — ok")
    return ok
# ────────────────────────────────────────────────────────────


def checar_bandas(arr_imagem: np.ndarray, arr_mascara: np.ndarray) -> bool:
    """
    Confirma que a imagem tem 3 bandas (RGB) e a máscara tem 1 banda.
    """
    n_img  = arr_imagem.shape[0]
    n_mask = arr_mascara.shape[0]
    ok_img  = n_img  == 3
    ok_mask = n_mask == 1

    print("\n── Número de bandas ────────────────────────")
    print(f"  Imagem : {n_img}  (esperado: 3)  {'✓' if ok_img  else '✗'}")
    print(f"  Máscara: {n_mask} (esperado: 1)  {'✓' if ok_mask else '✗'}")
    return ok_img and ok_mask


def checar_valores_mascara(arr_mascara: np.ndarray) -> bool:
    """
    Garante que a máscara só contém 0 e 1.

    A U-Net faz classificação binária pixel a pixel.
    Se houver valores fora de {0, 1}, a função de perda
    (binary_crossentropy) vai calcular errado.
    """
    valores = np.unique(arr_mascara)
    ok = set(valores).issubset({0.0, 1.0})

    print("\n── Valores únicos na máscara ───────────────")
    print(f"  Encontrados: {valores}")
    print(f"  {'✓ Apenas 0 e 1 — ok' if ok else '✗ Valores inesperados — checar antes de treinar.'}")
    return ok


# ── GAP 2 CORRIGIDO ─────────────────────────────────────────
# Antes: só checava NaN na imagem
# Agora: checa imagem E máscara separadamente
#
# Por que a máscara também pode ter NaN?
# Quando você exporta do QGIS com recorte, bordas que ficaram
# fora da área anotada podem virar NaN em vez de 0.
# Se a máscara tiver NaN, o Dice Loss calcula errado.
def checar_nan(arr_imagem: np.ndarray, arr_mascara: np.ndarray) -> bool:
    """
    Verifica se há pixels inválidos (NaN) na imagem ou na máscara.

    NaN significa 'Not a Number' — é como um buraco no dado.
    Pode aparecer em imagens Sentinel-2 nas bordas de cena,
    ou na máscara quando exportada com área não coberta.
    A rede neural não consegue aprender com NaN.
    """
    n_nan_img  = int(np.isnan(arr_imagem).sum())
    n_nan_mask = int(np.isnan(arr_mascara).sum())
    ok = (n_nan_img == 0) and (n_nan_mask == 0)

    print("\n── Pixels NaN ──────────────────────────────")
    print(f"  Imagem : {'✓ Nenhum NaN' if n_nan_img  == 0 else f'✗ {n_nan_img} pixel(s) NaN!'}")
    print(f"  Máscara: {'✓ Nenhum NaN' if n_nan_mask == 0 else f'✗ {n_nan_mask} pixel(s) NaN!'}")
    return ok
# ────────────────────────────────────────────────────────────


def checar_tipo_dado(arr_imagem: np.ndarray, arr_mascara: np.ndarray):
    """
    Informa o tipo de dado dos arrays (uint16, float32...).

    - uint16  → valores 0–65535  (Sentinel-2 original)
    - float32 → já normalizado   (ideal para o modelo)

    A normalização para [0, 1] será feita no Módulo 2.
    """
    print("\n── Tipo de dado ────────────────────────────")
    print(f"  Imagem : {arr_imagem.dtype}")
    print(f"  Máscara: {arr_mascara.dtype}")
    print("  → A imagem será convertida para float32 no Módulo 2.")


def relatorio_estatisticas(arr_imagem: np.ndarray):
    """
    Mostra min/max/média de cada banda da imagem.

    Isso ajuda a entender a faixa de valores antes de normalizar.
    Foi aqui que descobrimos que o máximo real era ~2700,
    não 10000 — decisão importante para a normalização.
    (Função informativa — não rejeita o par.)
    """
    print("\n── Estatísticas por banda ──────────────────")
    for i in range(arr_imagem.shape[0]):
        banda = arr_imagem[i]
        print(
            f"  Banda {i+1}: "
            f"min={banda.min()}, "
            f"max={banda.max()}, "
            f"média={banda.mean():.1f}"
        )


def validar_dataset():
    """
    Função principal: roda os checks em todos os pares
    encontrados automaticamente nas pastas configuradas.
    """
    print("=" * 50)
    print("  MÓDULO 1 — Validação dos dados")
    print("=" * 50)

    pares_validos = []

    for i, par in enumerate(PARES, start=1):
        print(f"\n{'='*50}")
        print(f"  Par {i}: {os.path.basename(par['imagem'])}")
        print(f"{'='*50}")

        with rasterio.open(par["imagem"]) as ds_img, \
             rasterio.open(par["mascara"]) as ds_mask:

            arr_img  = ds_img.read()
            arr_mask = ds_mask.read()

            print(f"\nArquivos carregados:")
            print(f"  Imagem  — shape: {arr_img.shape}  dtype: {arr_img.dtype}")
            print(f"  Máscara — shape: {arr_mask.shape} dtype: {arr_mask.dtype}")

            resultados = {
                "CRS iguais"               : checar_crs(ds_img, ds_mask),
                "Extensão geográfica ok"   : checar_extensao_geografica(ds_img, ds_mask),
                "Resolução compatível"     : checar_resolucao(ds_img, ds_mask),
                "Dimensões 128×128"        : checar_dimensoes(arr_img, arr_mask),
                "Bandas corretas"          : checar_bandas(arr_img, arr_mask),
                "Valores máscara ok"       : checar_valores_mascara(arr_mask),
                "Sem pixels NaN"           : checar_nan(arr_img, arr_mask),
            }

            checar_tipo_dado(arr_img, arr_mask)
            relatorio_estatisticas(arr_img)

        print(f"\n── Resumo do Par {i} ────────────────────────")
        par_ok = True
        for nome, passou in resultados.items():
            print(f"  {'✓' if passou else '✗'}  {nome}")
            if not passou:
                par_ok = False

        if par_ok:
            print(f"\n  ✓ Par {i} ok — adicionado à lista.")
            pares_validos.append(par)
        else:
            print(f"\n  ✗ Par {i} tem problema — não será usado no treino.")

    # Resumo geral
    print(f"\n{'='*50}")
    print(f"  RESULTADO FINAL")
    print(f"{'='*50}")
    print(f"  Pares válidos : {len(pares_validos)} / {len(PARES)}")
    if len(pares_validos) == len(PARES):
        print("  ✓ Tudo ok! Pode avançar para o Módulo 2.")
    else:
        print("  ✗ Corrija os pares com problema antes de continuar.")
    print("=" * 50)

    return pares_validos


# Chamada no final da célula
pares_validos = validar_dataset()


# Módulo 2 — Pré-processamento

In [ ]:
# ============================================================
# MÓDULO 2 — Transposição e normalização
# ============================================================
# Objetivo: converter cada par (imagem + máscara) de uint16
# bruto para float32 normalizado, pronto para o Módulo 3.
#
# Recebe : pares_validos  (lista de dicts com caminhos .tif)
# Retorna: pares_processados (lista de dicts com arrays numpy)
# ============================================================

import numpy as np
import rasterio


def transpor(arr: np.ndarray) -> np.ndarray:
    """
    Muda a ordem dos eixos de (Bandas, H, W) para (H, W, Bandas).

    O rasterio lê imagens no formato (B, H, W) — banda na frente.
    O Keras espera (H, W, B) — banda no final.
    É só uma reorganização, nenhum pixel é alterado.
    """
    return np.transpose(arr, (1, 2, 0))


def normalizar(imagem: np.ndarray) -> np.ndarray:
    """
    Divide todos os pixels por 10000 e limita ao intervalo [0, 1].

    O Sentinel-2 entrega valores uint16 com range típico de ~0–3000.
    O valor fixo 10000 é o teto oficial da reflectância de superfície
    do produto L2A — garante que todas as imagens usem a mesma régua,
    independente dos valores reais de cada cena.

    np.clip garante que nenhum valor vaze acima de 1.0 caso
    algum pixel anômalo tenha valor acima de 10000.
    """
    return np.clip(imagem / 10000.0, 0.0, 1.0)


def preprocessar(pares_validos: list) -> list:
    """
    Função principal: percorre todos os pares válidos,
    aplica transposição e normalização, e retorna os arrays prontos.
    """
    print("=" * 50)
    print("  MÓDULO 2 — Transposição e normalização")
    print("=" * 50)

    pares_processados = []

    for i, par in enumerate(pares_validos, start=1):
        with rasterio.open(par["imagem"]) as ds:
            imagem = ds.read().astype(np.float32)   # (3, 128, 128)

        with rasterio.open(par["mascara"]) as ds:
            mascara = ds.read().astype(np.float32)  # (1, 128, 128)

        print(f"\n  Par {i}:")
        print(f"  ANTES  | shape: {imagem.shape} | min: {imagem.min():.1f} | max: {imagem.max():.1f}")

        imagem  = normalizar(transpor(imagem))   # (128, 128, 3) float32 [0,1]
        mascara = transpor(mascara)              # (128, 128, 1) float32 {0,1}

        print(f"  DEPOIS | shape: {imagem.shape} | min: {imagem.min():.4f} | max: {imagem.max():.4f}")

        pares_processados.append({
            "imagem" : imagem,
            "mascara": mascara,
        })

    print(f"\n  ✓ {len(pares_processados)} par(es) prontos para o Módulo 3.")
    print("=" * 50)

    return pares_processados

# Módulo 3 - Dataset


In [ ]:
# ============================================================
# MÓDULO 3 — Dataset e augmentation
# ============================================================
# Recebe : pares_processados (lista de dicts com arrays numpy)
# Retorna: ds_treino, ds_val, ds_teste (tf.data.Dataset)
# ============================================================

import numpy as np
import tensorflow as py

# ----------------------------------------------------------
# CONFIGURAÇÃO
# ----------------------------------------------------------
SPLIT_TREINO = 0.70  # 70% treino
SPLIT_VAL    = 0.15  # 15% validação
SPLIT_TESTE  = 0.15  # 15% teste  ← abre só no final!
BATCH_SIZE   = 2
SEED         = 42
# ----------------------------------------------------------


def aplicar_augmentation(imagem: np.ndarray, mascara: np.ndarray) -> list:
    """
    Gera 16 variações do par (imagem, máscara).
    Aplicada SOMENTE nos pares de treino.
    """
    pares_aug = []

    def add(img, mask):
        pares_aug.append((img.copy(), mask.copy()))

    # Geométricas
    add(imagem,                               mascara)
    add(np.fliplr(imagem),                    np.fliplr(mascara))
    add(np.flipud(imagem),                    np.flipud(mascara))
    add(np.rot90(imagem, k=1),                np.rot90(mascara, k=1))
    add(np.rot90(imagem, k=2),                np.rot90(mascara, k=2))
    add(np.rot90(imagem, k=3),                np.rot90(mascara, k=3))
    add(np.rot90(np.fliplr(imagem), k=1),     np.rot90(np.fliplr(mascara), k=1))
    add(np.rot90(np.flipud(imagem), k=1),     np.rot90(np.flipud(mascara), k=1))

    # Brilho — só na imagem, máscara não muda
    for fator in [1.1, 0.9]:
        img_b = np.clip(imagem * fator, 0.0, 1.0)
        add(img_b,                          mascara)
        add(np.fliplr(img_b),               np.fliplr(mascara))
        add(np.flipud(img_b),               np.flipud(mascara))
        add(np.rot90(img_b, k=1),           np.rot90(mascara, k=1))

    return pares_aug


def montar_dataset(pares_processados: list):
    """
    Função principal:
    1. Divide os pares ORIGINAIS em treino/validação/teste
    2. Aplica augmentation SÓ nos pares de treino
    3. Validação e teste usam imagens originais, sem augmentation
    """
    print("=" * 52)
    print("  MÓDULO 3 — Dataset e augmentation")
    print("=" * 52)

    # ── SPLIT NOS PARES ORIGINAIS — antes de qualquer augmentation
    np.random.seed(SEED)
    indices = np.random.permutation(len(pares_processados))

    n_treino = int(len(pares_processados) * SPLIT_TREINO)
    n_val    = int(len(pares_processados) * SPLIT_VAL)
    # teste pega o resto — garante que todos os pares são usados
    # ⚠ não abrir ds_teste até terminar o desenvolvimento!

    idx_treino = indices[:n_treino]
    idx_val    = indices[n_treino : n_treino + n_val]
    idx_teste  = indices[n_treino + n_val:]

    pares_treino = [pares_processados[i] for i in idx_treino]
    pares_val    = [pares_processados[i] for i in idx_val]
    pares_teste  = [pares_processados[i] for i in idx_teste]

    print(f"\n  Total de pares    : {len(pares_processados)}")
    print(f"  Pares de treino   : {len(pares_treino)}  (70%)")
    print(f"  Pares de validação: {len(pares_val)}  (15%)")
    print(f"  Pares de teste    : {len(pares_teste)}  (15%) ← abrir só no final!")
    print(f"  (split feito antes do augmentation — sem data leakage)")

    # ── AUGMENTATION SÓ NO TREINO
    imgs_treino, masks_treino = [], []

    for par in pares_treino:
        variacoes = aplicar_augmentation(par["imagem"], par["mascara"])
        for img_aug, mask_aug in variacoes:
            imgs_treino.append(img_aug)
            masks_treino.append(mask_aug)

    print(f"\n  Augmentation (×16) aplicado nos {len(pares_treino)} pares de treino")
    print(f"  Total exemplos treino: {len(imgs_treino)}")

    # ── VALIDAÇÃO E TESTE SEM AUGMENTATION — imagens originais
    imgs_val   = [par["imagem"]  for par in pares_val]
    masks_val  = [par["mascara"] for par in pares_val]
    imgs_teste = [par["imagem"]  for par in pares_teste]
    masks_teste= [par["mascara"] for par in pares_teste]

    print(f"  Total exemplos val  : {len(imgs_val)}  (originais)")
    print(f"  Total exemplos teste: {len(imgs_teste)}  (originais)")

    # Converte para arrays numpy
    X_treino = np.array(imgs_treino,  dtype=np.float32)
    y_treino = np.array(masks_treino, dtype=np.float32)
    X_val    = np.array(imgs_val,     dtype=np.float32)
    y_val    = np.array(masks_val,    dtype=np.float32)
    X_teste  = np.array(imgs_teste,   dtype=np.float32)
    y_teste  = np.array(masks_teste,  dtype=np.float32)

    print(f"\n  X_treino — shape: {X_treino.shape}")
    print(f"  X_val    — shape: {X_val.shape}")
    print(f"  X_teste  — shape: {X_teste.shape}")

    # Embaralha só o treino
    idx_shuffle = np.random.permutation(len(X_treino))
    X_treino = X_treino[idx_shuffle]
    y_treino = y_treino[idx_shuffle]

    # Monta tf.data.Dataset
    ds_treino = (
        tf.data.Dataset
        .from_tensor_slices((X_treino, y_treino))
        .shuffle(buffer_size=len(X_treino), seed=SEED)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    ds_val = (
        tf.data.Dataset
        .from_tensor_slices((X_val, y_val))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    ds_teste = (
        tf.data.Dataset
        .from_tensor_slices((X_teste, y_teste))
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

    print(f"\n  Batch size        : {BATCH_SIZE}")
    print(f"  Batches treino    : {len(ds_treino)}")
    print(f"  Batches validação : {len(ds_val)}")
    print(f"  Batches teste     : {len(ds_teste)}")
    print("\n" + "=" * 52)
    print("  ✓ Dataset pronto. Próximo: Módulo 4 — U-Net.")
    print("=" * 52)

    return ds_treino, ds_val, ds_teste, pares_teste


# Módulo 4 - Modelo_Unet


In [ ]:
"""
MÓDULO 4 — Arquitetura U-Net padrão (Keras)
============================================
Objetivo: construir a U-Net, compilar e treinar usando os
datasets gerados no Módulo 3.

O que é a U-Net?
  É uma rede neural com formato de "U". A metade esquerda
  (encoder) comprime a imagem progressivamente para entender
  o contexto ("o que é isso?"). A metade direita (decoder)
  expande de volta para o tamanho original para localizar
  exatamente onde está ("onde exatamente?").
  As "skip connections" são pontes que conectam cada nível
  do encoder com o mesmo nível do decoder — preservando
  detalhes de borda que seriam perdidos na compressão.

Entrada : (128, 128, 3)  — imagem RGB normalizada
Saída   : (128, 128, 1)  — máscara binária prevista

Pré-requisito: ter rodado o Módulo 3 (ds_treino, ds_val).
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ----------------------------------------------------------
# CONFIGURAÇÃO
# ----------------------------------------------------------
EPOCHS      = 50     # número de épocas de treino
ALTURA      = 128
LARGURA     = 128
CANAIS      = 3      # RGB
# ----------------------------------------------------------


def bloco_convolucional(x, filtros: int):
    """
    Bloco básico da U-Net: duas convoluções seguidas de ReLU.

    Analogia: cada convolução é como um filtro fotográfico que
    realça um padrão específico (bordas, texturas, cores).
    Dois filtros em sequência = detector mais sofisticado.

    filtros: quantos padrões diferentes o bloco vai aprender.
    Dobra a cada nível do encoder (32→64→128→256).
    """
    x = layers.Conv2D(filtros, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)   # estabiliza o treino
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filtros, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    return x


def encoder_block(x, filtros: int):
    """
    Um nível do encoder: convolução + MaxPooling.

    O MaxPooling reduz a imagem pela metade (128→64→32→16).
    É como olhar a imagem de mais longe para ver o contexto.

    Retorna:
        skip : saída antes do pooling (guardada para a skip connection)
        x    : saída após o pooling (segue para o próximo nível)
    """
    skip = bloco_convolucional(x, filtros)
    x    = layers.MaxPooling2D(2)(skip)
    return skip, x


def decoder_block(x, skip, filtros: int):
    """
    Um nível do decoder: upsample + concatena skip + convolução.

    O UpSampling2D dobra a imagem de volta (16→32→64→128).
    A concatenação com o skip traz de volta os detalhes de borda
    que foram perdidos durante o encoder.

    É como reconstruir uma foto pixelada usando a versão original
    como referência.
    """
    x = layers.UpSampling2D(2)(x)
    x = layers.Concatenate()([x, skip])   # skip connection
    x = bloco_convolucional(x, filtros)
    return x


def construir_unet(altura=128, largura=128, canais=3):
    """
    Monta a U-Net completa.

    Estrutura:
      Encoder: 4 níveis (32 → 64 → 128 → 256 filtros)
      Bottleneck: 512 filtros (ponto mais comprimido)
      Decoder: 4 níveis espelhando o encoder
      Saída: 1 canal com sigmoid (probabilidade 0–1 por pixel)
    """
    entradas = keras.Input(shape=(altura, largura, canais))

    # ── Encoder (descida do U) ──────────────────────────
    # Cada nível guarda um "skip" para reconectar no decoder
    skip1, x = encoder_block(entradas, 32)   # 128×128 → 64×64
    skip2, x = encoder_block(x, 64)          # 64×64  → 32×32
    skip3, x = encoder_block(x, 128)         # 32×32  → 16×16
    skip4, x = encoder_block(x, 256)         # 16×16  → 8×8

    # ── Bottleneck (fundo do U) ─────────────────────────
    # Ponto de maior compressão — entende o contexto global
    x = bloco_convolucional(x, 512)          # 8×8

    # ── Decoder (subida do U) ───────────────────────────
    x = decoder_block(x, skip4, 256)         # 8×8   → 16×16
    x = decoder_block(x, skip3, 128)         # 16×16 → 32×32
    x = decoder_block(x, skip2, 64)          # 32×32 → 64×64
    x = decoder_block(x, skip1, 32)          # 64×64 → 128×128

    # ── Saída ───────────────────────────────────────────
    # Conv 1×1 com sigmoid: cada pixel vira uma probabilidade
    # de ser plantação (0 = fundo, 1 = plantação)
    saidas = layers.Conv2D(1, 1, activation="sigmoid")(x)

    modelo = keras.Model(entradas, saidas, name="unet")
    return modelo


def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    Coeficiente Dice — métrica principal para segmentação.

    Mede a sobreposição entre a máscara prevista e a real.
    Valor 1.0 = sobreposição perfeita.
    Valor 0.0 = nenhuma sobreposição.

    Por que não usar só acurácia?
    Porque nossa máscara tem muito mais pixels 0 (fundo) do que
    1 (plantação). Um modelo que prevê tudo como 0 teria ~90%
    de acurácia mas seria inútil. O Dice penaliza isso.
    """
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersecao = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersecao + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


def dice_loss(y_true, y_pred):
    """
    Loss baseada no Dice (1 - Dice coefficient).
    Quanto menor, melhor o modelo está segmentando.
    """
    return 1.0 - dice_coefficient(y_true, y_pred)


def loss_combinada(y_true, y_pred):
    """
    Combina Dice Loss + Binary Crossentropy.

    Por que combinar?
    - Dice Loss é ótima para lidar com classes desbalanceadas
      (muito fundo, pouca plantação).
    - Binary Crossentropy é estável no início do treino quando
      o Dice ainda oscila muito.
    Juntas convergem mais rápido e de forma mais estável.
    """
    return dice_loss(y_true, y_pred) + keras.losses.binary_crossentropy(
        y_true, y_pred
    )


def compilar_e_treinar(ds_treino, ds_val):
    """
    Constrói, compila e treina a U-Net.
    Retorna o modelo treinado e o histórico de métricas.
    """
    print("=" * 52)
    print("  MÓDULO 4 — U-Net")
    print("=" * 52)

    # 1. Constrói o modelo
    modelo = construir_unet(ALTURA, LARGURA, CANAIS)
    print(f"\nModelo construído.")
    print(f"  Parâmetros treináveis: "
          f"{modelo.count_params():,}".replace(",", "."))

    # 2. Compila
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=loss_combinada,
        metrics=[dice_coefficient, "accuracy"],
    )
    print("  Compilado com Adam (lr=1e-4) + loss combinada.")

    # 3. Callbacks — ações automáticas durante o treino
    callbacks = [
        # Salva o melhor modelo automaticamente
        keras.callbacks.ModelCheckpoint(
            filepath="/content/drive/MyDrive/teste_drive/unet_melhor.keras",
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        # Para o treino se não melhorar por 15 épocas seguidas
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient",
            mode="max",
            patience=15,
            verbose=1,
            restore_best_weights=True,
        ),
        # Reduz o learning rate se estiver estagnado
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_dice_coefficient",
            mode="max",
            factor=0.5,       # divide o lr por 2
            patience=7,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    # 4. Treina
    print(f"\nIniciando treino — {EPOCHS} épocas máximas...")
    print("  (EarlyStopping pode parar antes se convergir)\n")

    historico = modelo.fit(
        ds_treino,
        validation_data=ds_val,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )

    print("\n" + "=" * 52)
    print("  ✓ Treino concluído.")
    print("  Melhor modelo salvo em: unet_melhor.keras")
    print("  Próximo passo: Módulo 5 — avaliação e predição.")
    print("=" * 52)

    return modelo, historico

#if __name__ == "__main__":
    # Para testar isolado sem o Módulo 3:
    #modelo = construir_unet()
    #modelo.summary()

# Módulo 5 - Predição

In [ ]:
# ============================================================
# MÓDULO 5 — Avaliação visual e predição
# ============================================================
# Objetivo: carregar o modelo salvo, rodar a predição nos pares
# de TESTE (separados no Módulo 3) e mostrar lado a lado:
#   - Imagem original (RGB)
#   - Máscara real (ground truth)
#   - Máscara prevista pelo modelo
#   - Sobreposição (imagem + máscara prevista)
#
# Pré-requisito: ter rodado os Módulos 2, 3 e 4.
# Recebe: pares_teste (lista de dicts com arrays numpy)
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# ----------------------------------------------------------
# CONFIGURAÇÃO
# ----------------------------------------------------------
MODELO_PATH    = "/content/drive/MyDrive/teste_drive/unet_melhor.keras"
RESULTADO_PATH = "/content/drive/MyDrive/teste_drive/predicoes_unet.png"
LIMIAR         = 0.5
# ----------------------------------------------------------


def calcular_metricas(mascara_real: np.ndarray, mascara_prevista: np.ndarray):
    """
    Calcula Dice e IoU entre a máscara real e a prevista.
    """
    real = mascara_real.flatten()
    prev = mascara_prevista.flatten()

    intersecao = np.sum(real * prev)
    uniao      = np.sum(real) + np.sum(prev) - intersecao

    dice = (2 * intersecao + 1e-6) / (np.sum(real) + np.sum(prev) + 1e-6)
    iou  = (intersecao + 1e-6) / (uniao + 1e-6)

    return dice, iou


def visualizar_predicoes(modelo, pares_teste: list):
    """
    Para cada par de teste: roda a predição e plota 4 painéis.
    Usa os arrays numpy que já vêm prontos do Módulo 2.
    """
    n = len(pares_teste)
    fig, eixos = plt.subplots(n, 4, figsize=(16, 4 * n))
    fig.suptitle(
        f"Resultados da U-Net — {n} pares de teste",
        fontsize=14, y=1.01
    )

    # Garante que eixos sempre é bidimensional (mesmo com 1 par)
    if n == 1:
        eixos = np.expand_dims(eixos, axis=0)

    dices = []
    ious  = []

    for i, par in enumerate(pares_teste):
        # Arrays já normalizados vindos do Módulo 2
        imagem       = par["imagem"]   # (128, 128, 3)
        mascara_real = par["mascara"]  # (128, 128, 1)

        # Predição
        entrada     = np.expand_dims(imagem, axis=0)
        prob_map    = modelo.predict(entrada, verbose=0)[0, :, :, 0]
        mascara_bin = (prob_map > LIMIAR).astype(np.float32)

        dice, iou = calcular_metricas(mascara_real[:, :, 0], mascara_bin)
        dices.append(dice)
        ious.append(iou)

        # Painel 1 — Imagem RGB
        eixos[i, 0].imshow(imagem)
        eixos[i, 0].set_title(f"Par {i+1} — Imagem RGB")
        eixos[i, 0].axis("off")

        # Painel 2 — Máscara real
        eixos[i, 1].imshow(mascara_real[:, :, 0], cmap="gray", vmin=0, vmax=1)
        eixos[i, 1].set_title("Máscara real")
        eixos[i, 1].axis("off")

        # Painel 3 — Probabilidade bruta
        im = eixos[i, 2].imshow(prob_map, cmap="RdYlGn", vmin=0, vmax=1)
        eixos[i, 2].set_title(
            f"Probabilidade prevista\nDice={dice:.3f}  IoU={iou:.3f}"
        )
        eixos[i, 2].axis("off")
        plt.colorbar(im, ax=eixos[i, 2], fraction=0.046, pad=0.04)

        # Painel 4 — Sobreposição
        eixos[i, 3].imshow(imagem)
        overlay = np.zeros((*mascara_bin.shape, 4))
        overlay[mascara_bin == 1] = [0, 1, 0, 0.4]
        eixos[i, 3].imshow(overlay)
        eixos[i, 3].set_title(f"Sobreposição (limiar={LIMIAR})")
        eixos[i, 3].axis("off")

        print(f"  Par {i+1}: Dice={dice:.4f}  IoU={iou:.4f}")

    # Resumo final
    print(f"\n  Média — Dice={np.mean(dices):.4f}  IoU={np.mean(ious):.4f}")

    plt.tight_layout()
    plt.savefig(RESULTADO_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n  Imagem salva em: {RESULTADO_PATH}")


def avaliar(pares_teste: list):
    """
    Função principal: carrega o modelo e roda a avaliação
    apenas nos pares de teste separados no Módulo 3.
    """
    print("=" * 52)
    print("  MÓDULO 5 — Avaliação visual")
    print("=" * 52)
    print(f"\n  Avaliando {len(pares_teste)} pares de teste.")
    print("  ⚠ Esses pares nunca foram vistos pelo modelo.")

    print(f"\n  Carregando modelo: {MODELO_PATH}")
    modelo = keras.models.load_model(
        MODELO_PATH,
        custom_objects={
            "dice_coefficient": lambda y_true, y_pred: (
                2 * tf.reduce_sum(
                    tf.reshape(y_true, [-1]) * tf.reshape(y_pred, [-1])
                ) + 1e-6
            ) / (
                tf.reduce_sum(tf.reshape(y_true, [-1])) +
                tf.reduce_sum(tf.reshape(y_pred, [-1])) + 1e-6
            ),
            "loss_combinada": lambda y_true, y_pred: (
                1.0 - (
                    2 * tf.reduce_sum(
                        tf.reshape(y_true, [-1]) * tf.reshape(y_pred, [-1])
                    ) + 1e-6
                ) / (
                    tf.reduce_sum(tf.reshape(y_true, [-1])) +
                    tf.reduce_sum(tf.reshape(y_pred, [-1])) + 1e-6
                )
            ) + keras.losses.binary_crossentropy(y_true, y_pred),
        }
    )
    print("  ✓ Modelo carregado.")

    print("\n  Gerando predições...")
    visualizar_predicoes(modelo, pares_teste)

    print("\n" + "=" * 52)
    print("  ✓ Avaliação concluída.")
    print("  O gráfico foi salvo no seu Drive.")
    print("=" * 52)

    return modelo



# Módulo 6 -

In [ ]:
# ============================================================
# MÓDULO 6 — Análise e documentação de métricas
# ============================================================
# Objetivo: gerar os gráficos de análise do treino e dos
# resultados nos pares de teste para documentação da IC.
#
# Gráficos gerados:
#   1. Curva de Loss (treino vs validação por época)
#   2. Curva de Dice (treino vs validação por época)
#   3. Tabela com Dice e IoU por par de teste
#   4. Gráfico de barras — Dice e IoU por par
#
# Pré-requisito: ter rodado os Módulos 3, 4 e 5.
# Recebe:
#   historico   — objeto retornado pelo Módulo 4
#   pares_teste — lista retornada pelo Módulo 3
#   modelo      — modelo retornado pelo Módulo 5
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras

# ----------------------------------------------------------
# CONFIGURAÇÃO
# ----------------------------------------------------------
RESULTADO_PATH = "/content/drive/MyDrive/teste_drive/analise_unet.png"
LIMIAR         = 0.5
# ----------------------------------------------------------


def calcular_metricas(mascara_real: np.ndarray, mascara_prevista: np.ndarray):
    """
    Calcula Dice e IoU entre a máscara real e a prevista.
    Mesma função do Módulo 5 — replicada aqui para este módulo
    ser independente.
    """
    real = mascara_real.flatten()
    prev = mascara_prevista.flatten()

    intersecao = np.sum(real * prev)
    uniao      = np.sum(real) + np.sum(prev) - intersecao

    dice = (2 * intersecao + 1e-6) / (np.sum(real) + np.sum(prev) + 1e-6)
    iou  = (intersecao + 1e-6) / (uniao + 1e-6)

    return dice, iou


def coletar_metricas_teste(modelo, pares_teste: list):
    """
    Roda a predição nos pares de teste e coleta Dice e IoU
    de cada par. Retorna duas listas prontas para plotar.
    """
    dices = []
    ious  = []

    for par in pares_teste:
        imagem       = par["imagem"]
        mascara_real = par["mascara"]

        entrada     = np.expand_dims(imagem, axis=0)
        prob_map    = modelo.predict(entrada, verbose=0)[0, :, :, 0]
        mascara_bin = (prob_map > LIMIAR).astype(np.float32)

        dice, iou = calcular_metricas(mascara_real[:, :, 0], mascara_bin)
        dices.append(dice)
        ious.append(iou)

    return dices, ious


def plotar_curva_loss(ax, historico):
    """
    Gráfico 1 — Curva de Loss por época.

    Mostra loss de treino e validação lado a lado.
    O que queremos ver: as duas curvas caindo juntas.
    Se val_loss subir enquanto loss cai = overfitting.
    """
    epocas = range(1, len(historico.history["loss"]) + 1)

    ax.plot(epocas, historico.history["loss"],
            color="#2196F3", linewidth=2, label="Treino")
    ax.plot(epocas, historico.history["val_loss"],
            color="#F44336", linewidth=2, linestyle="--", label="Validação")

    ax.set_title("Curva de Loss por Época", fontsize=13, fontweight="bold")
    ax.set_xlabel("Época")
    ax.set_ylabel("Loss (Dice Loss + BCE)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Marca a época com menor val_loss
    melhor_epoca = np.argmin(historico.history["val_loss"]) + 1
    melhor_val   = min(historico.history["val_loss"])
    ax.axvline(x=melhor_epoca, color="gray", linestyle=":", alpha=0.7)
    ax.annotate(
        f"  melhor\n  época {melhor_epoca}\n  val_loss={melhor_val:.4f}",
        xy=(melhor_epoca, melhor_val),
        fontsize=8, color="gray"
    )


def plotar_curva_dice(ax, historico):
    """
    Gráfico 2 — Curva de Dice por época.

    Mostra Dice de treino e validação lado a lado.
    O que queremos ver: as duas curvas subindo juntas em direção a 1.0.
    """
    epocas = range(1, len(historico.history["dice_coefficient"]) + 1)

    ax.plot(epocas, historico.history["dice_coefficient"],
            color="#2196F3", linewidth=2, label="Treino")
    ax.plot(epocas, historico.history["val_dice_coefficient"],
            color="#F44336", linewidth=2, linestyle="--", label="Validação")

    ax.set_title("Curva de Dice por Época", fontsize=13, fontweight="bold")
    ax.set_xlabel("Época")
    ax.set_ylabel("Dice Coefficient")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Marca a época com maior val_dice
    melhor_epoca = np.argmax(historico.history["val_dice_coefficient"]) + 1
    melhor_val   = max(historico.history["val_dice_coefficient"])
    ax.axvline(x=melhor_epoca, color="gray", linestyle=":", alpha=0.7)
    ax.annotate(
        f"  melhor\n  época {melhor_epoca}\n  val_dice={melhor_val:.4f}",
        xy=(melhor_epoca, melhor_val),
        fontsize=8, color="gray"
    )


def plotar_barras(ax, dices, ious):
    """
    Gráfico 3 — Barras de Dice e IoU por par de teste.

    Cada par aparece com duas barras lado a lado (Dice e IoU).
    Linha horizontal marca a média de cada métrica.
    """
    n_pares = len(dices)
    x       = np.arange(n_pares)
    largura = 0.35

    barras_dice = ax.bar(x - largura/2, dices, largura,
                         color="#4CAF50", alpha=0.85, label="Dice")
    barras_iou  = ax.bar(x + largura/2, ious,  largura,
                         color="#FF9800", alpha=0.85, label="IoU")

    # Linhas de média
    ax.axhline(y=np.mean(dices), color="#4CAF50",
               linestyle="--", linewidth=1.5,
               label=f"Média Dice={np.mean(dices):.4f}")
    ax.axhline(y=np.mean(ious), color="#FF9800",
               linestyle="--", linewidth=1.5,
               label=f"Média IoU={np.mean(ious):.4f}")

    ax.set_title("Dice e IoU por Par de Teste", fontsize=13, fontweight="bold")
    ax.set_xlabel("Par de Teste")
    ax.set_ylabel("Valor da Métrica")
    ax.set_xticks(x)
    ax.set_xticklabels([f"Par {i+1}" for i in range(n_pares)], rotation=45)
    ax.set_ylim(0.8, 1.02)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")


def plotar_tabela(ax, dices, ious):
    """
    Gráfico 4 — Tabela com valores exatos de Dice e IoU.

    Apresentação limpa para copiar direto na IC.
    Última linha mostra a média geral.
    """
    ax.axis("off")

    # Monta os dados da tabela
    linhas = [[f"Par {i+1}", f"{dices[i]:.4f}", f"{ious[i]:.4f}"]
              for i in range(len(dices))]
    linhas.append(["Média", f"{np.mean(dices):.4f}", f"{np.mean(ious):.4f}"])

    colunas = ["Par", "Dice", "IoU"]

    tabela = ax.table(
        cellText=linhas,
        colLabels=colunas,
        loc="center",
        cellLoc="center"
    )
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(10)
    tabela.scale(1.2, 1.6)

    # Destaca o cabeçalho
    for j in range(3):
        tabela[0, j].set_facecolor("#37474F")
        tabela[0, j].set_text_props(color="white", fontweight="bold")

    # Destaca a linha de média
    ultima = len(linhas)
    for j in range(3):
        tabela[ultima, j].set_facecolor("#E3F2FD")
        tabela[ultima, j].set_text_props(fontweight="bold")

    ax.set_title("Resultados por Par de Teste", fontsize=13,
                 fontweight="bold", pad=20)


def analisar(historico, pares_teste: list, modelo):
    """
    Função principal: gera todos os gráficos de análise
    e salva em um único arquivo PNG no Drive.
    """
    print("=" * 52)
    print("  MÓDULO 6 — Análise de métricas")
    print("=" * 52)

    # Coleta métricas dos pares de teste
    print("\n  Coletando métricas dos pares de teste...")
    dices, ious = coletar_metricas_teste(modelo, pares_teste)

    # Imprime tabela no terminal
    print("\n  Resultados por par:")
    print("  " + "-" * 36)
    for i, (d, iou) in enumerate(zip(dices, ious)):
        print(f"  Par {i+1:>2}: Dice={d:.4f}  IoU={iou:.4f}")
    print("  " + "-" * 36)
    print(f"  Média — Dice={np.mean(dices):.4f}  IoU={np.mean(ious):.4f}")

    # Monta figura com 4 gráficos
    print("\n  Gerando gráficos...")
    fig = plt.figure(figsize=(18, 14))
    fig.suptitle(
        "U-Net — Análise Completa de Métricas\n"
        "Segmentação de Lavouras Agrícolas — Sentinel-2",
        fontsize=15, fontweight="bold", y=0.98
    )

    gs = gridspec.GridSpec(2, 2, figure=fig,
                           hspace=0.4, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])  # Curva de Loss
    ax2 = fig.add_subplot(gs[0, 1])  # Curva de Dice
    ax3 = fig.add_subplot(gs[1, 0])  # Barras por par
    ax4 = fig.add_subplot(gs[1, 1])  # Tabela

    plotar_curva_loss(ax1, historico)
    plotar_curva_dice(ax2, historico)
    plotar_barras(ax3, dices, ious)
    plotar_tabela(ax4, dices, ious)

    plt.savefig(RESULTADO_PATH, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\n  Gráfico salvo em: {RESULTADO_PATH}")
    print("\n" + "=" * 52)
    print("  ✓ Análise concluída.")
    print("=" * 52)


# ----------------------------------------------------------
# CHAMADA — rode esta célula após os Módulos 3, 4 e 5
# ----------------------------------------------------------
# analisar(historico, pares_teste, modelo)

# Nova seção

In [ ]:
# ============================================================
# PREDIÇÃO COMPLETA — Independente do pipeline principal
# ============================================================
# Objetivo: carregar o modelo salvo e rodar a predição em
# todas as imagens da pasta, mostrando lado a lado:
#   - Imagem RGB original
#   - Máscara real (ground truth)
#   - Máscara prevista pelo modelo
#
# Completamente independente — não precisa ter rodado
# nenhum outro módulo antes. Só precisa do modelo no Drive
# e das pastas de imagens e máscaras.
# ============================================================

import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# ----------------------------------------------------------
# CONFIGURAÇÃO — ajuste os caminhos se necessário
# ----------------------------------------------------------
PASTA_IMAGENS  = "/content/drive/MyDrive/ic/imagem"
PASTA_MASCARAS = "/content/drive/MyDrive/ic/mask"
MODELO_PATH    = "/content/drive/MyDrive/teste_drive/unet_melhor.keras"
RESULTADO_PATH = "/content/drive/MyDrive/teste_drive/predicao_completa.png"
LIMIAR         = 0.5
# ----------------------------------------------------------


def extrair_numero(nome_arquivo):
    """
    Extrai o número do nome do arquivo para ordenar corretamente.
    Ex: "10.tif" → 10, não "10" (string que colocaria 10 antes de 2)
    """
    try:
        return int(nome_arquivo.replace(".tif", ""))
    except ValueError:
        return -1


def carregar_pares():
    """
    Varre as pastas e monta a lista de todos os pares válidos.
    Só inclui arquivos que existem nas duas pastas.
    """
    imgs  = {f for f in os.listdir(PASTA_IMAGENS)  if f.endswith(".tif")}
    masks = {f for f in os.listdir(PASTA_MASCARAS) if f.endswith(".tif")}
    nomes = sorted(imgs & masks, key=extrair_numero)

    pares = []
    for nome in nomes:
        pares.append({
            "nome"   : nome,
            "imagem" : os.path.join(PASTA_IMAGENS,  nome),
            "mascara": os.path.join(PASTA_MASCARAS, nome),
        })

    print(f"  {len(pares)} par(es) encontrado(s).")
    return pares


def normalizar(arr_imagem: np.ndarray) -> np.ndarray:
    """
    Transpõe (B,H,W) → (H,W,B) e normaliza por 10000.
    Mesmo processo do Módulo 2 — garante que a imagem
    chega ao modelo no formato que ele foi treinado.

    ⚠ Sempre normaliza por 10000 — nunca pelo máximo da imagem.
    """
    arr = np.transpose(arr_imagem, (1, 2, 0)).astype(np.float32)
    arr = np.clip(arr / 10000.0, 0.0, 1.0)
    return arr


def carregar_modelo():
    """
    Carrega o modelo salvo no Drive com as funções customizadas.
    As funções precisam ser passadas aqui porque foram definidas
    fora do Keras — ele não as reconhece automaticamente.
    """
    def dice_coefficient(y_true, y_pred, smooth=1e-6):
        y_true_f = tf.reshape(y_true, [-1])
        y_pred_f = tf.reshape(y_pred, [-1])
        intersecao = tf.reduce_sum(y_true_f * y_pred_f)
        return (2. * intersecao + smooth) / (
            tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
        )

    def loss_combinada(y_true, y_pred):
        dice_loss = 1.0 - dice_coefficient(y_true, y_pred)
        return dice_loss + keras.losses.binary_crossentropy(y_true, y_pred)

    modelo = keras.models.load_model(
        MODELO_PATH,
        custom_objects={
            "dice_coefficient": dice_coefficient,
            "loss_combinada"  : loss_combinada,
        }
    )
    print(f"  ✓ Modelo carregado: {MODELO_PATH}")
    return modelo


def rodar_predicao(modelo, pares: list):
    """
    Roda a predição em todos os pares e plota 3 painéis por imagem:
      Coluna 1 — Imagem RGB original
      Coluna 2 — Máscara real (ground truth)
      Coluna 3 — Máscara prevista pelo modelo

    Salva tudo num único PNG no Drive.
    """
    n = len(pares)

    # 3 colunas por imagem, n linhas
    fig, eixos = plt.subplots(n, 3, figsize=(12, 4 * n))
    fig.suptitle(
        f"Predição completa — {n} imagens\n"
        f"Modelo: {os.path.basename(MODELO_PATH)}",
        fontsize=14, fontweight="bold", y=1.01
    )

    # Garante que eixos é sempre bidimensional (mesmo com 1 par)
    if n == 1:
        eixos = np.expand_dims(eixos, axis=0)

    for i, par in enumerate(pares):
        print(f"  Processando {par['nome']} ({i+1}/{n})...")

        # Carrega imagem e máscara do disco
        with rasterio.open(par["imagem"]) as ds_img:
            arr_img = ds_img.read()
        with rasterio.open(par["mascara"]) as ds_mask:
            arr_mask = ds_mask.read()

        # Normaliza a imagem — mesmo processo do Módulo 2
        imagem_norm = normalizar(arr_img)          # (128, 128, 3)
        mascara_real = arr_mask[0].astype(np.float32)  # (128, 128)

        # Roda a predição
        entrada     = np.expand_dims(imagem_norm, axis=0)  # (1, 128, 128, 3)
        prob_map    = modelo.predict(entrada, verbose=0)[0, :, :, 0]
        mascara_bin = (prob_map > LIMIAR).astype(np.float32)

        nome_sem_ext = par["nome"].replace(".tif", "")

        # Painel 1 — Imagem RGB
        eixos[i, 0].imshow(imagem_norm)
        eixos[i, 0].set_title(f"{nome_sem_ext} — Imagem RGB", fontsize=9)
        eixos[i, 0].axis("off")

        # Painel 2 — Máscara real
        eixos[i, 1].imshow(mascara_real, cmap="gray", vmin=0, vmax=1)
        eixos[i, 1].set_title("Máscara real", fontsize=9)
        eixos[i, 1].axis("off")

        # Painel 3 — Máscara prevista
        # ⚠ Mostra a probabilidade bruta (não binarizada)
        # para você ver onde o modelo ficou inseguro (tons de cinza)
        # Pixels brancos = certeza de lavoura
        # Pixels pretos  = certeza de fundo
        # Pixels cinza   = região de dúvida → possível ruído
        im = eixos[i, 2].imshow(prob_map, cmap="RdYlGn", vmin=0, vmax=1)
        eixos[i, 2].set_title("Probabilidade prevista", fontsize=9)
        eixos[i, 2].axis("off")
        plt.colorbar(im, ax=eixos[i, 2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(RESULTADO_PATH, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\n  ✓ Imagem salva em: {RESULTADO_PATH}")


def predizer_tudo():
    """
    Função principal — carrega modelo e roda predição em tudo.
    """
    print("=" * 52)
    print("  PREDIÇÃO COMPLETA — todas as imagens")
    print("=" * 52)

    print("\n  Carregando modelo...")
    modelo = carregar_modelo()

    print("\n  Carregando pares...")
    pares = carregar_pares()

    print("\n  Rodando predições...")
    rodar_predicao(modelo, pares)

    print("\n" + "=" * 52)
    print("  ✓ Concluído.")
    print("=" * 52)


# ----------------------------------------------------------
# CHAMADA — rode esta célula sozinha, sem depender de nada
# ----------------------------------------------------------
predizer_tudo()

# Teste

In [ ]:
pares_validos = validar_dataset()



In [ ]:
pares_processados = preprocessar(pares_validos)

In [ ]:
ds_treino, ds_val, ds_teste, pares_teste = montar_dataset(pares_processados)

In [ ]:
modelo, historico = compilar_e_treinar(ds_treino, ds_val)

In [ ]:
modelo = avaliar(pares_teste)

In [ ]:
analisar(historico, pares_teste, modelo)

In [ ]:
# ============================================================
# TESTE ISOLADO — imagem nova sem máscara
# ============================================================

import numpy as np
import rasterio
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

MODELO_PATH = "/content/drive/MyDrive/teste_drive/unet_melhor.keras"
IMAGEM_PATH = "/content/101.tif"  # ← troque aqui
LIMIAR      = 0.5

# ── Passo 1: carrega o raster bruto e mostra os valores
with rasterio.open(IMAGEM_PATH) as ds:
    arr = ds.read().astype(np.float32)  # (3, 128, 128)

print("Valores brutos (antes de normalizar):")
for b in range(arr.shape[0]):
    print(f"  Banda {b+1}: min={arr[b].min():.0f}  max={arr[b].max():.0f}  média={arr[b].mean():.0f}")


   # ── Passo 2: normaliza por 10000 — igual ao Módulo 2
imagem = np.transpose(arr, (1, 2, 0))  # (128, 128, 3)
imagem = np.clip(imagem / 10000.0, 0.0, 1.0)

print("\nValores após normalização:")
for b in range(imagem.shape[2]):
    print(f"  Banda {b+1}: min={imagem[:,:,b].min():.3f}  max={imagem[:,:,b].max():.3f}  média={imagem[:,:,b].mean():.3f}")
# ── Passo 3: carrega o modelo
modelo = keras.models.load_model(
    MODELO_PATH,
    custom_objects={
        "dice_coefficient": lambda y_true, y_pred: (
            2 * tf.reduce_sum(
                tf.reshape(y_true, [-1]) * tf.reshape(y_pred, [-1])
            ) + 1e-6
        ) / (
            tf.reduce_sum(tf.reshape(y_true, [-1])) +
            tf.reduce_sum(tf.reshape(y_pred, [-1])) + 1e-6
        ),
        "loss_combinada": lambda y_true, y_pred: (
            1.0 - (
                2 * tf.reduce_sum(
                    tf.reshape(y_true, [-1]) * tf.reshape(y_pred, [-1])
                ) + 1e-6
            ) / (
                tf.reduce_sum(tf.reshape(y_true, [-1])) +
                tf.reduce_sum(tf.reshape(y_pred, [-1])) + 1e-6
            )
        ) + keras.losses.binary_crossentropy(y_true, y_pred),
    }
)
print("\n✓ Modelo carregado.")

# ── Passo 4: predição
entrada     = np.expand_dims(imagem, axis=0)       # (1, 128, 128, 3)
prob_map    = modelo.predict(entrada, verbose=0)[0, :, :, 0]
mascara_bin = (prob_map > LIMIAR).astype(np.float32)

print(f"  Probabilidade média prevista: {prob_map.mean():.3f}")
print(f"  Pixels classificados como lavoura: {mascara_bin.sum():.0f} / {128*128}")

# ── Passo 5: visualização
fig, eixos = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Teste com imagem nova", fontsize=13)

eixos[0].imshow(imagem)
eixos[0].set_title("Imagem RGB (normalizada)")
eixos[0].axis("off")

im = eixos[1].imshow(prob_map, cmap="RdYlGn", vmin=0, vmax=1)
eixos[1].set_title("Probabilidade prevista")
eixos[1].axis("off")
plt.colorbar(im, ax=eixos[1], fraction=0.046, pad=0.04)

eixos[2].imshow(imagem)
overlay = np.zeros((*mascara_bin.shape, 4))
overlay[mascara_bin == 1] = [0, 1, 0, 0.4]
eixos[2].imshow(overlay)
eixos[2].set_title(f"Sobreposição (limiar={LIMIAR})")
eixos[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import rasterio
import json

PASTA_IMAGENS = "/content/drive/MyDrive/ic/imagem"

def extrair_numero(nome):
    try:
        return int(nome.replace(".tif", ""))
    except ValueError:
        return None

nomes = sorted(
    [f for f in os.listdir(PASTA_IMAGENS) if f.endswith(".tif") and extrair_numero(f) is not None],
    key=extrair_numero
)

patches = []
for nome in nomes:
    path = os.path.join(PASTA_IMAGENS, nome)
    with rasterio.open(path) as ds:
        b = ds.bounds
        crs = str(ds.crs)
        patches.append({
            "id": extrair_numero(nome),
            "nome": nome,
            "crs": crs,
            "minx": b.left,
            "miny": b.bottom,
            "maxx": b.right,
            "maxy": b.top
        })

with open("/content/drive/MyDrive/ic/patches_bounds.json", "w") as f:
    json.dump(patches, f, indent=2)

print(f"✓ {len(patches)} patches exportados")
print(f"Primeiro: {patches[0]}")
print(f"Último: {patches[-1]}")